In [5]:
#[CELL 1] STUDENT CONFIGURATION

from pathlib import Path
import csv
import json
import shutil

LAST_NAME = "Bautista"
STUDENT_ID = "TUPM-26-3506"

SEED_DIGIT = int(STUDENT_ID[-1])

TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1

print("Target OS:", TARGET_OS)
print("Target Behavior Class:", TARGET_BEHAVIOR_CLASS)

Target OS: Android
Target Behavior Class: 2


In [6]:
# [CELL 2] WORKSPACE SETUP

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

In [7]:
# [CELL 3] DATASET PATH

dataset_path = DATA_DIR / "user_behavior_dataset.csv"

export_csv = OUTPUT_DIR / (
    f"{LAST_NAME}_{STUDENT_ID}_mobile_profile.csv"
)

summary_json = OUTPUT_DIR / (
    f"{LAST_NAME}_{STUDENT_ID}_mobile_summary.json"
)

backup_csv = BACKUP_DIR / export_csv.name

In [8]:
# [CELL 4] FILTER DATASET

selected_records = []

total_records = 0
os_records = 0

battery_total = 0
data_total = 0

with open(dataset_path, "r", encoding="utf-8") as file:

    reader = csv.DictReader(file)

    for row in reader:

        total_records += 1

        if row["Operating System"] != TARGET_OS:
            continue

        os_records += 1

        try:
            behavior_class = int(
                row["User Behavior Class"]
            )

            if behavior_class != TARGET_BEHAVIOR_CLASS:
                continue

            battery = float(
                row["Battery Drain (mAh/day)"]
            )

            data_usage = float(
                row["Data Usage (MB/day)"]
            )

        except ValueError:
            continue

        battery_total += battery
        data_total += data_usage

        selected_records.append({
            "User ID": row["User ID"],
            "Device Model": row["Device Model"],
            "Operating System": row["Operating System"],
            "Battery Drain (mAh/day)": battery,
            "Data Usage (MB/day)": data_usage
        })

print("Total records:", total_records)
print("OS matches:", os_records)
print(
    "Behavior matches:",
    len(selected_records)
)

Total records: 700
OS matches: 554
Behavior matches: 114


In [9]:
# [CELL 5] CALCULATE AVERAGES OR MEANS

selected_count = len(selected_records)

if selected_count > 0:

    avg_battery = (
        battery_total / selected_count
    )

    avg_data = (
        data_total / selected_count
    )

else:

    avg_battery = 0
    avg_data = 0

print(
    "Average Battery Drain:",
    round(avg_battery, 2)
)

print(
    "Average Data Usage:",
    round(avg_data, 2)
)

Average Battery Drain: 887.11
Average Data Usage: 451.44


In [10]:
# [CELL 6] EXPORTING

with open(
    export_csv,
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=[
            "User ID",
            "Device Model",
            "Operating System",
            "Battery Drain (mAh/day)",
            "Data Usage (MB/day)"
        ]
    )

    writer.writeheader()

    for record in selected_records:
        writer.writerow(record)

print("Created:", export_csv)

Created: output\Bautista_TUPM-26-3506_mobile_profile.csv


In [11]:
# [CELL 7] VERIFICATION

exported_count = 0

with open(export_csv, "r", encoding="utf-8") as file:

    reader = csv.DictReader(file)

    for row in reader:
        exported_count += 1

count_verified = (
    exported_count == selected_count
)

print("Selected:", selected_count)
print("Exported:", exported_count)
print("Verified:", count_verified)

Selected: 114
Exported: 114
Verified: True


In [12]:
# [CELL 8] SUMMARY JSON

summary = {

    "operating_system": TARGET_OS,

    "behavior_class":
    TARGET_BEHAVIOR_CLASS,

    "total_records":
    total_records,

    "records_matching_os":
    os_records,

    "records_in_target_class":
    selected_count,

    "average_battery_drain":
    round(avg_battery, 2),

    "average_data_usage":
    round(avg_data, 2),

    "count_verification":
    count_verified,

    "exported_csv":
    export_csv.name
}

with open(
    summary_json,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        summary,
        file,
        indent=4
    )

print("Created:", summary_json)

Created: output\Bautista_TUPM-26-3506_mobile_summary.json


In [13]:
# [CELL 9] BACKUP

shutil.copy2(export_csv, backup_csv)

backup_size = backup_csv.stat().st_size

print("Backup:", backup_csv)
print("Size:", backup_size, "bytes")

Backup: backup\Bautista_TUPM-26-3506_mobile_profile.csv
Size: 4573 bytes


In [14]:
# [CELL 10] ASSESSMENT OUTPUT

print("\nMOBILE DEVICE USAGE PROFILE")
print("-" * 40)

print(
    "OS / Behavior:",
    TARGET_OS,
    TARGET_BEHAVIOR_CLASS
)

print("Total Records:", total_records)

print(
    "Records Matching OS:",
    os_records
)

print(
    "Records in Target Class:",
    selected_count
)

print(
    "Average Battery:",
    round(avg_battery,2),
    "mAh/day"
)

print(
    "Average Data Usage:",
    round(avg_data,2),
    "MB/day"
)

print("Count Verification:",
    count_verified)

print("CSV File:",
    export_csv.name)

print("JSON File:",
    summary_json.name)

print(
    "Backup Size:",
    backup_size,
    "bytes"
)


MOBILE DEVICE USAGE PROFILE
----------------------------------------
OS / Behavior: Android 2
Total Records: 700
Records Matching OS: 554
Records in Target Class: 114
Average Battery: 887.11 mAh/day
Average Data Usage: 451.44 MB/day
Count Verification: True
CSV File: Bautista_TUPM-26-3506_mobile_profile.csv
JSON File: Bautista_TUPM-26-3506_mobile_summary.json
Backup Size: 4573 bytes
